<a href="https://colab.research.google.com/github/Innocent-I/Smart-Irrigation-ML/blob/main/notebooks/02_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Smart Irrigation ML — Data Preprocessing

This notebook prepares the smart irrigation sensor dataset for machine learning.

## Objectives

- Load and validate the dataset
- Examine unusual sensor observations
- Define input features and target variable
- Split the dataset into training and testing sets
- Prepare the data for machine learning models

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load dataset directly from GitHub

data_url = "https://raw.githubusercontent.com/Innocent-I/Smart-Irrigation-ML/main/data/raw/smart_irrigation_data.csv"

df = pd.read_csv(data_url)

print("Dataset loaded successfully from GitHub!")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully from GitHub!
Dataset shape: (3000, 4)


,Soil Moisture,Temperature,Air Humidity,Pump Data
0,683.802906,29.184908,71.789699,0
1,408.571567,33.707205,77.977391,1
2,659.092074,24.760311,60.776282,1
3,842.929764,32.738515,59.323543,0
4,414.199320,25.692744,66.624914,1


In [3]:
# Investigate potential outliers in Soil Moisture using the IQR method

Q1 = df["Soil Moisture"].quantile(0.25)
Q3 = df["Soil Moisture"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

soil_outliers = df[
    (df["Soil Moisture"] < lower_bound) |
    (df["Soil Moisture"] > upper_bound)
]

print("\nNumber of potential Soil Moisture outliers:", len(soil_outliers))

display(soil_outliers)

Q1: 501.36257545
Q3: 821.8674228250001
IQR: 320.50484737500005
Lower bound: 20.605304387499928
Upper bound: 1302.6246938875001

Number of potential Soil Moisture outliers: 0


,Soil Moisture,Temperature,Air Humidity,Pump Data


In [4]:
# Define input features (X) and target variable (y)

X = df[["Soil Moisture", "Temperature", "Air Humidity"]]
y = df["Pump Data"]

print("Input features (X):")
display(X.head())

print("\nTarget variable (y):")
display(y.head())

print("\nX shape:", X.shape)
print("y shape:", y.shape)

Input features (X):


,Soil Moisture,Temperature,Air Humidity
0,683.802906,29.184908,71.789699
1,408.571567,33.707205,77.977391
2,659.092074,24.760311,60.776282
3,842.929764,32.738515,59.323543
4,414.199320,25.692744,66.624914



Target variable (y):


,Pump Data
0,0
1,1
2,1
3,0
4,1



X shape: (3000, 3)
y shape: (3000,)


In [5]:
from sklearn.model_selection import train_test_split

# Split dataset: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training features: (2400, 3)
Testing features: (600, 3)

Training target distribution:
Pump Data
1    1255
0    1145
Name: count, dtype: int64

Testing target distribution:
Pump Data
1    314
0    286
Name: count, dtype: int64


In [6]:
import numpy as np
from sklearn.metrics import accuracy_score

# Find the best soil-moisture threshold using TRAINING data only
thresholds = np.linspace(
    X_train["Soil Moisture"].min(),
    X_train["Soil Moisture"].max(),
    500
)

best_threshold = None
best_accuracy = 0

for threshold in thresholds:
    predictions = (X_train["Soil Moisture"] < threshold).astype(int)
    accuracy = accuracy_score(y_train, predictions)

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_threshold = threshold

print("Best Soil Moisture Threshold:", round(best_threshold, 2))
print("Training Accuracy:", round(best_accuracy, 4))

Best Soil Moisture Threshold: 682.58
Training Accuracy: 0.9988


In [7]:
# Evaluate the baseline threshold on unseen test data

baseline_predictions = (
    X_test["Soil Moisture"] < best_threshold
).astype(int)

baseline_test_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

print("Baseline Test Accuracy:", round(baseline_test_accuracy, 4))

Baseline Test Accuracy: 0.9983


In [8]:
from sklearn.preprocessing import StandardScaler

# Create scaler
scaler = StandardScaler()

# Learn scaling parameters from training data only
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same transformation to test data
X_test_scaled = scaler.transform(X_test)

print("Training data scaled:", X_train_scaled.shape)
print("Testing data scaled:", X_test_scaled.shape)

print("\nTraining feature means after scaling:")
print(np.round(X_train_scaled.mean(axis=0), 4))

print("\nTraining feature standard deviations after scaling:")
print(np.round(X_train_scaled.std(axis=0), 4))

Training data scaled: (2400, 3)
Testing data scaled: (600, 3)

Training feature means after scaling:
[ 0. -0.  0.]

Training feature standard deviations after scaling:
[1. 1. 1.]


## Preprocessing Summary

- The dataset contains 3,000 complete observations with no missing or duplicate records.
- The IQR analysis did not identify statistical outliers in the Soil Moisture variable.
- All observations were therefore retained.
- Three predictor variables were selected: Soil Moisture, Temperature, and Air Humidity.
- Pump Data was defined as the binary target variable.
- The dataset was divided into 80% training and 20% testing data using stratified sampling.
- A simple soil-moisture threshold baseline was established using training data only.
- The optimal training threshold was approximately 682.58, achieving 99.88% training accuracy and 99.83% test accuracy.
- Feature scaling was fitted only on the training data to avoid data leakage.

## Key Observation

The extremely high performance of the simple soil-moisture threshold suggests that pump status in this dataset is largely determined by the raw soil-moisture measurement. Machine learning models will therefore be evaluated against this strong baseline rather than assuming that a more complex model is automatically better.

## Next Step

The next notebook will train and compare classification models, including Logistic Regression, Decision Tree, and Random Forest, against the simple threshold baseline.